In [1]:
import torch


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/.venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/.venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1080, in launch_instance
    app.start()
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/

In [2]:
words = ['anna','bob']

In [3]:
b = {}   #dictionarry to store all the training example

for word in words:
    chs = '.' + word + '.'
    for first, second in zip(chs, chs[1:]):

        if (first, second) not in b:
            b[(first, second)] = 1
        else :
            b[(first, second)] += 1

print(b)

{('.', 'a'): 1, ('a', 'n'): 1, ('n', 'n'): 1, ('n', 'a'): 1, ('a', '.'): 1, ('.', 'b'): 1, ('b', 'o'): 1, ('o', 'b'): 1, ('b', '.'): 1}


In [4]:
map = {
    '.' : 0,
    'a' : 1,
    'b' : 2,
    'n' : 3,
    'o' : 4,
}


In [6]:
mat = torch.zeros((5,5))

print(mat)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])


In [121]:
p = torch.tensor([0.7, 0.2, 0.1])
print(torch.multinomial(p, num_samples=1))

tensor([0])


In [122]:
words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))

# We use '.' as the special boundary token.
# It represents both the beginning and end of a name.

stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi['.'] = 0

itos = {i: ch for ch, i in stoi.items()}

print("Vocabulary:")
print(stoi)

Vocabulary:
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}


In [123]:
# ---------------------------------------------------------
# 3. Create the count matrix
# ---------------------------------------------------------

N = torch.zeros((len(stoi), len(stoi)), dtype=torch.int32)


# ---------------------------------------------------------
# 4. Count all character transitions
# ---------------------------------------------------------

for word in words:

    # Add boundary tokens
    chs = ['.'] + list(word) + ['.']

    # Create consecutive character pairs
    for ch1, ch2 in zip(chs, chs[1:]):

        # Convert characters → integer IDs
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]

        # Increment the corresponding count
        N[ix1, ix2] += 1

In [124]:
# ---------------------------------------------------------
# 5. Convert counts → probabilities
# ---------------------------------------------------------

# Sum each row.
# Each row represents one current character.
row_sums = N.sum(dim=1, keepdim=True)

# Normalize each row.
P = N.float() / row_sums

In [125]:
# ---------------------------------------------------------
# 6. Generate new names
# ---------------------------------------------------------

generator = torch.Generator().manual_seed(42)

num_names = 20

for _ in range(num_names):

    # Start with the boundary token '.'
    ix = 0

    generated_name = []

    while True:

        # Probability distribution for the next character
        p = P[ix]

        # Sample the next character
        ix = torch.multinomial(
            p,
            num_samples=1,
            generator=generator
        ).item()

        # If we generated '.', the name is finished
        if ix == 0:
            break

        # Convert integer ID → character
        generated_name.append(itos[ix])

    print("".join(generated_name))

ya
syahavilin
dleekahmangonya
tryahe
chen
ena
da
amiiae
a
keles
ly
a
oy
asityi
pepolannezale
shahlamion
nacelucyanarivieriaquten
kigshmole
ei
tonylyan


In [126]:


# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

words = open("names.txt", "r").read().splitlines()


# --------------------------------------------------
# 2. Vocabulary
# --------------------------------------------------

chars = sorted(list(set("".join(words))))

stoi = {
    '<START>': 0,
    '<END>': 1,
}

for i, ch in enumerate(chars, start=2):
    stoi[ch] = i

itos = {i: ch for ch, i in stoi.items()}


# --------------------------------------------------
# 3. Trigram count tensor
# --------------------------------------------------

N = torch.zeros(
    (len(stoi), len(stoi), len(stoi)),
    dtype=torch.int32
)


# --------------------------------------------------
# 4. Count trigrams
# --------------------------------------------------

for word in words:

    chs = ['<START>', '<START>'] + list(word) + ['<END>']

    for ch1, ch2, ch3 in zip(
        chs,
        chs[1:],
        chs[2:]
    ):

        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]

        N[ix1, ix2, ix3] += 1


# --------------------------------------------------
# 5. Convert counts → probabilities
# --------------------------------------------------

counts = N.sum(dim=2, keepdim=True)

P = N.float() / counts


# --------------------------------------------------
# 6. Generate names
# --------------------------------------------------

g = torch.Generator().manual_seed(42)

for _ in range(20):

    ix1 = stoi['<START>']
    ix2 = stoi['<START>']

    out = []

    while True:

        # P(next | previous, current)
        p = P[ix1, ix2]

        # Sample next character
        ix3 = torch.multinomial(
            p,
            num_samples=1,
            generator=g
        ).item()

        # End of name
        if ix3 == stoi['<END>']:
            break

        # Add character
        out.append(itos[ix3])

        # Shift context
        ix1 = ix2
        ix2 = ix3

    print("".join(out))

kannlee
leen
maksanshikaevicamaka
va
bleigda
deko
lihadanajerlayani
sa
lin
stilotonna
mey
jen
ruzeelleen
abdomie
nelluwan
taes
marcartli
ludhanney
bren
sriya
